## Key Performance Indicators

In [ ]:
import geopandas as gpd
import rasterio as rio
import matplotlib as plt
import rioxarray as rxr
import numpy as np
import pandas as pd


base_path = '/data/LEON/P6_forest_agri_uganda/'

## KPI generation

KPI 0 – Forest stock baseline (ha)

KPI 1 – Deforestation rate (ha yr⁻¹ + ratio of remaining forest) 

KPI 2 – Forest→agriculture ratio (ha yr⁻¹ + ratio of deforestation)

In [28]:


# ---------------- LOAD DATA ----------------
county = gpd.read_file(base_path + "area/counties_bugoma_cut_32636.gpkg")

defo = rxr.open_rasterio(base_path + "s1_deforested/s1_deforestation_year_20-24.tif")
forest_mask = rxr.open_rasterio(base_path + "area/forest_extent_bugoma_2020_32636.tif")
crop = rxr.open_rasterio(base_path + "crops/meancrops_24-25_32636.tif")

# ---------------- ALIGN GRIDS TO S1 GRID ----------------
forest_mask = forest_mask.rio.reproject_match(defo)
crop = crop.rio.reproject_match(defo)

# ---------------- PIXEL AREA (ha) ----------------
px_w, px_h = defo.rio.resolution()
pixel_area_ha = abs(px_w * px_h) / 10000.0

results = []

# ---------------- KPI LOOP ----------------
for _, row in county.iterrows():
    geom = [row.geometry]
    name = row["ADM4_EN"]

    # Clip all rasters
    d_clip = defo.rio.clip(geom, county.crs).squeeze().values
    f_clip = forest_mask.rio.clip(geom, county.crs).squeeze().values
    c_clip = crop.rio.clip(geom, county.crs).squeeze().values

    # Convert S1 fractional timestamp → integer year
    d_year = np.floor(d_clip).astype(np.int32)

    # Forest baseline (2020)
    forest_ha = (f_clip == 1).sum() * pixel_area_ha

    data = {
        "County": name,
        "Forest_2020_ha": round(float(forest_ha), 2)
    }

    total_loss = 0
    total_crop = 0

    # Loop through individual years
    for year in [2021, 2022, 2023, 2024]:

        # Deforestation inside 2020 forest
        loss_mask = (d_year == year) & (f_clip == 1)
        loss_ha = loss_mask.sum() * pixel_area_ha

        # Loss → cropland (crop prob > 0.1)
        crop_mask = loss_mask & (c_clip > 0.1)
        crop_ha = crop_mask.sum() * pixel_area_ha

        # Annual deforestation rate
        rate = (loss_ha / forest_ha * 100) if forest_ha > 0 else 0

        # Store results
        data[f"Loss_{year}_ha"] = round(float(loss_ha), 2)
        data[f"Rate_{year}_%"] = round(float(rate), 3)
        data[f"Crop_{year}_ha"] = round(float(crop_ha), 2)
        data[f"Crop_{year}_%"] = round(float((crop_ha / loss_ha * 100) if loss_ha > 0 else 0), 2)

        total_loss += loss_ha
        total_crop += crop_ha

    # Totals across 2021–2024
    data["Total_Loss_ha"] = round(float(total_loss), 2)
    data["Total_Crop_ha"] = round(float(total_crop), 2)
    data["KPI_2_Total_Crop_%"] = round(float((total_crop / total_loss * 100) if total_loss > 0 else 0), 2)

    # Average annual deforestation rate
    data["Avg_Annual_Rate_%"] = round(float((total_loss / 4) / forest_ha * 100) if forest_ha > 0 else 0, 3)

    results.append(data)

df = pd.DataFrame(results)
print(df)

df.to_csv("forest_kpis_yearly.csv", index=False)

/tmp/ipykernel_120030/3883436024.py:29: RuntimeWarning: invalid value encountered in cast
  d_year = np.floor(d_clip).astype(np.int32)
/tmp/ipykernel_120030/3883436024.py:29: RuntimeWarning: invalid value encountered in cast
  d_year = np.floor(d_clip).astype(np.int32)


      County  Forest_2020_ha  Loss_2021_ha  Rate_2021_%  Crop_2021_ha  \
0    Kabwoya        31159.68        603.00        1.935        436.59   
1  Kyangwali        13032.44        718.02        5.509        584.64   

   Crop_2021_%  Loss_2022_ha  Rate_2022_%  Crop_2022_ha  Crop_2022_%  ...  \
0        72.40       1334.52        4.283       1128.78        84.58  ...   
1        81.42       1794.06       13.766       1528.56        85.20  ...   

   Crop_2023_ha  Crop_2023_%  Loss_2024_ha  Rate_2024_%  Crop_2024_ha  \
0        854.01        92.34        348.48        1.118        273.78   
1       1238.85        89.20        585.36        4.492        467.91   

   Crop_2024_%  Total_Loss_ha  Total_Crop_ha  KPI_2_Total_Crop_%  \
0        78.56        3210.84        2693.16               83.88   
1        79.94        4486.32        3819.96               85.15   

   Avg_Annual_Rate_%  
0              2.576  
1              8.606  

[2 rows x 22 columns]
